In [1]:
import numpy as np
import pandas as pd

## Coding a spam classifier with naive Bayes

#### Loading the dataset

In [2]:
url = "https://raw.githubusercontent.com/luisguiserrano/manning/master/Chapter_08_Naive_Bayes/emails.csv"
emails = pd.read_csv(url)
emails.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


_____
#### Pre-processing the email

We use the split() function to turn
the words into a list. We check only whether each word appears in
the email, regardless of how many times it appears, so we turn it
into a set and then into a list

In [3]:
def process_email(text):
    text = text.lower()
    return list(set(text.split())) 

emails["words"] = emails["text"].apply(process_email)

In [4]:
emails[:10]

,text,spam,words
0,Subject: naturally irresistible your corporate...,1,"[marketing, effective, that, with, done, subje..."
1,Subject: the stock trading gunslinger fanny i...,1,"[stock, morristown, pirogue, optima, incredibl..."
2,Subject: unbelievable new homes made easy im ...,1,"[unbelievable, ask, $, that, hearing, from, ap..."
3,Subject: 4 color printing special request add...,1,"[printing, additional, advertisement, 338, spe..."
4,"Subject: do not have money , get software cds ...",1,"[from, yet, with, d, get, subject:, ,, to, alo..."
5,"Subject: great nnews hello , welcome to medzo...",1,"[ntiaiity, 150, l, confidant, r, ag, miilion, ..."
6,Subject: here ' s a hot play in motion homela...,1,"[stock, that, ega, cost, exchange, objectives,..."
7,Subject: save your money buy getting this thin...,1,"[ship, can, that, want, yet, lot, with, buy, e..."
8,Subject: undeliverable : home based business f...,1,"[name, i, 2002, undeliverable, subject:, 5, 09..."
9,Subject: save your money buy getting this thin...,1,"[lasts, can, ship, that, want, yet, lot, with,..."


____
#### Calculating the prior (probability that a random email is spam)

For this, we simply divide the number of spam emails by the total number of emails in the dataset.

In [6]:
num_emails = len(emails)
num_spam = sum(emails['spam'])

print("Number of emails:", num_emails)
print("Number of spam emails:", num_spam)
print()

# Calculating the prior probability that an email is spam
print("Probability of spam:", num_spam/num_emails)

Number of emails: 5728
Number of spam emails: 1368

Probability of spam: 0.2388268156424581


_____
#### Training a naive Bayes model

Our plan is to write a dictionary, and in this dictionary record every word, and its pair of occurrences in spam and ham. We also use Laplace smoothing, which prevents zero counts.

In [5]:
model = {}

for index, email in emails.iterrows():
    for word in email["words"]:
        if word not in model:
            model[word] = {"spam": 1, "ham": 1}
        if word in model:
            if email["spam"]:
                model[word]["spam"] += 1
            else:
                model[word]["ham"] += 1

In [6]:
model["lottery"]

{'spam': 9, 'ham': 1}

In [7]:
model["sale"]

{'spam': 39, 'ham': 42}

In [8]:
print(model)

{'marketing': {'spam': 148, 'ham': 99}, 'effective': {'spam': 66, 'ham': 96}, 'that': {'spam': 533, 'ham': 2471}, 'with': {'spam': 654, 'ham': 2663}, 'done': {'spam': 81, 'ham': 229}, 'subject:': {'spam': 1369, 'ham': 4361}, 'shouldn': {'spam': 28, 'ham': 9}, '100': {'spam': 141, 'ham': 118}, 'you': {'spam': 983, 'ham': 3477}, 'not': {'spam': 611, 'ham': 1568}, 'no': {'spam': 395, 'ham': 624}, 'logo': {'spam': 97, 'ham': 13}, 'all': {'spam': 472, 'ham': 1206}, '.': {'spam': 1337, 'ham': 4323}, 'and': {'spam': 989, 'ham': 3569}, 'budget': {'spam': 34, 'ham': 55}, 'have': {'spam': 634, 'ham': 2800}, 'lt': {'spam': 28, 'ham': 1}, 'specially': {'spam': 31, 'ham': 6}, 'through': {'spam': 148, 'ham': 495}, 'our': {'spam': 623, 'ham': 1492}, 'identity': {'spam': 81, 'ham': 4}, 'stylish': {'spam': 15, 'ham': 1}, 'really': {'spam': 72, 'ham': 268}, 'automaticaily': {'spam': 6, 'ham': 1}, 'hand': {'spam': 82, 'ham': 68}, 'drafts': {'spam': 25, 'ham': 3}, 'look': {'spam': 140, 'ham': 786}, 'is': 

The counts start at 1 (to avoid zero counts), so *lottery* appears in 8 spam emails and 0 ham emails, and *sale* appears in 38 spam emails and 41 ham emails.

#### Using the model to make predictions

In [9]:
def predict_bayes(word):
    word = word.lower()
    num_spam_with_word = model[word]["spam"]
    num_ham_with_word = model[word]["ham"]
    return num_spam_with_word / (num_spam_with_word + num_ham_with_word)

In [10]:
predict_bayes("lottery")

0.9

In [11]:
predict_bayes("sale")

0.48148148148148145

#### Naive Bayes Prediction


In [12]:
def predict_naive_bayes(email):
    total = len(emails)
    num_spam = sum(emails["spam"])  # Count spam emails
    num_ham = total - num_spam      # Count ham emails

    email = email.lower()           # Convert to lowercase
    words = set(email.split())      # Extract unique words

    spams = [1.0]                   # Starting spam score
    hams = [1.0]                    # Starting ham score

    for word in words:
        if word in model:           # Only use words seen during training
            spams.append(model[word]["spam"] / num_spam)  # Word probability given spam
            hams.append(model[word]["ham"] / num_ham)     # Word probability given ham

    # Multiply word probabilities by class counts (class priors)
    prod_spams = float(np.prod(spams) * num_spam)
    prod_hams = float(np.prod(hams) * num_ham)

    # Normalize to get the estimated probability of spam
    return prod_spams / (prod_spams + prod_hams)

1. **Combine word probabilities:** Multiply the probabilities of the words in the email, assuming they are conditionally independent:

2. **Calculate class scores:** Multiply the word probabilities by their class priors:

   **Why?** The words tell us how strongly the email resembles each class, while the priors account for how common each class already is. Combining them gives a score that considers both the email's content and the overall class distribution.

5. **Normalize:** Divide the spam score by the sum of both scores:
   This converts the scores into an estimated probability between 0 and 1.

Notice that the model predicts high probability for the messages that sound spammy, and low probabilities for those that sound legitimate.

In [13]:
# Spammy
predict_naive_bayes('lottery sale')

0.9638144470140118

In [14]:
# Legit
predict_naive_bayes('Hi mom how are you')

0.1255435886716446

In [15]:
# Legit, but with unknown words
predict_naive_bayes('Hi MOM how aRe yoU afdjsaklfsdhgjasdhfjklsd')

0.1255435886716446

In [16]:
# Legit
predict_naive_bayes('meet me at the lobby of the hotel at nine am')

6.964603508395963e-05

In [17]:
# Spammy
predict_naive_bayes('enter the lottery to win three million dollars')

0.9995234218677428

In [18]:
#Spammy
predict_naive_bayes('buy cheap lottery easy money now')

0.999973472265966

In [19]:
# Legit (hopefully?) :)
predict_naive_bayes('Grokking Machine Learning by Luis Serrano')

0.41971076454845674

In [20]:
# Noise (which the model takes as an empty message).
predict_naive_bayes('asdfgh')

0.2388268156424581

In [21]:
# For the empty message, the probability is simply the prior
# (i.e., the number of spam emails divided by the total number of emails in the dataset)
predict_naive_bayes('')

0.2388268156424581

#### Worked examples from the chapter

These cells reproduce the hand calculations in the chapter. The expected outputs are in the comments.

In [25]:
# Medical example: 1,000,000 people, 1 in 10,000 sick, test 99% accurate
sick = 1_000_000 / 10_000
healthy = 1_000_000 - sick
sick_positive = 0.99 * sick        # 99
healthy_positive = 0.01 * healthy  # 9,999
print(round(sick_positive / (sick_positive + healthy_positive), 4))  # 0.0098

0.0098
